# Load Datasets to S3
- May 2026
- Numantic Solutions (numanticsolutions.com)

In [1]:
import os, sys
import pandas as pd
import json

# AWS Python
import boto3

# Tools for loading data to S3
import s3_data_load as sdl


## Read local data


In [2]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

# Read local data into Pandas dataframes
df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))


## Clean up docs

## Load some document metadata

In [3]:
doc_metadata_file ="doc_metadata.json"
data_path = "../data/rag_eval_dataset"

with open(os.path.join(data_path,doc_metadata_file), "r", encoding="utf-8") as file:
    doc_metadata = json.load(file)


## Add metadata attributes and supplement text as needed

In [4]:
# Metadata columns
df_docs["source_type"] = df_docs["source_url"].map(doc_metadata["source_type_map"])
df_docs["title"] = df_docs["source_url"].map(doc_metadata["source_title_map"])

# Missing text
for key in doc_metadata["text_additions"].keys():
    doc_index = int(key.replace("doc_", ""))
    df_docs.loc[doc_index, "text"] = "{}\n\n{}".format(df_docs.loc[1, "text"],
                                                       doc_metadata["text_additions"][key])


In [5]:
print("Source documents")
display(df_docs.head())
# display(pd.DataFrame(df_docs["source_type"].value_counts()))

# print("Single-passage questions")
# display(df_spqs.head())


Source documents


,index,source_url,text,source_type,title
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,gaming,Bullet Kin
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,gaming,The Paths through the Underground/Underdark
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,data_science,Semantic and Textual Inference Chatbot Interfa...
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,data_science,LLMware
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,recipes,Building Block Recipes


## Load documents to S3

In [7]:

### Step 1. Set up session
region_name = "us-east-2"
profile_name = "ns-admin"
session = boto3.Session(profile_name=profile_name,
                        region_name=region_name)

### Step 2. Export documents to S3 in Bedrock-friendly format
s3_client = session.client('s3')
bucket = 'rag-search-tests'
s3_prefix = 'documents/'

### Step 3. Delete existing documents first
sdl.delete_bucket_files(s3_client=s3_client,
                        bucket=bucket,
                        s3_prefix=s3_prefix)

### Step 4. Upload text and metadata files
text_col = "text"
metadata_cols = ["source_url", "source_type", "title"]

sdl.process_and_upload(df=df_docs,
                       s3_client=s3_client,
                       bucket=bucket,
                       text_col=text_col,
                       metadata_cols=metadata_cols,
                       key_prefix=s3_prefix)


Deleting existing documents from rag-search-tests/documents/...
✓ Existing documents deleted
Successfully uploaded 20 documents and metadata files to prefix 'documents/'.
